# Elevation, in pure Python

`peaknav.terrain` answers *how high is this point* from the same compressed ASTER
dataset the app renders — summit heights corrected against surveyed values. No
renderer, no Java, no display — one dependency (Pillow) and a network connection the
first time each area is needed.

The dataset packs each 4×4 block of tiles into one ~30 MB archive; the first question
about an area downloads and unpacks it, and every later question in that area is
answered locally.

In [ ]:
from peaknav.terrain import elevation_at

elevation_at(45.9417, 7.7480)   # the Breithorn, above Zermatt

## Summit heights are corrected — but coordinates still matter

Raw ASTER is a stereo DEM: each pixel is about 30 m across, and a sharp rock spire
does not fill one, so spires used to read hundreds of metres low. The dataset now
corrects summits against surveyed elevations — but at ~30 m pixels, the summit can
still sit a pixel or two away from the coordinate *you* have for the peak, so the
value at the exact coordinate can run a few tens of metres under the survey. The
next section shows how to find the true top.

In [ ]:
peaks = [
    ("Breithorn",    45.9417,  7.7480, 4164),   # broad snow dome, reads true
    ("Matterhorn",   45.9763,  7.6586, 4478),   # spire; raw ASTER read ~4040 here
    ("Mont Blanc",   45.8326,  6.8652, 4808),
    ("Ben Nevis",    56.7969, -5.0036, 1345),
    ("Mount Rainier", 46.8523, -121.7603, 4392),
]

for name, lat, lon, surveyed in peaks:
    measured = elevation_at(lat, lon)
    print(f"{name:<14} {measured:>5} m   (surveyed {surveyed} m, "
          f"{measured - surveyed:+d})")

## Finding the true top

The residuals above are no longer the DEM clipping the spire — they are the summit
pixel sitting a nudge away from the coordinate we asked about. Scan a few arcseconds
around the coordinate and the corrected summit appears: the Matterhorn comes back
4484 m, the surveyed 4478 to within the encoding's 4 m step. (The scan is ~160
lookups; after the first cell above, the tiles are local, so it takes seconds.)

In [ ]:
def summit(lat, lon, span=0.002, step=0.0003):
    """The highest reading within ~span degrees of a coordinate, and where it is."""
    best = (0, lat, lon)
    steps = int(span / step)
    for i in range(-steps, steps + 1):
        for j in range(-steps, steps + 1):
            la, lo = lat + i * step, lon + j * step
            best = max(best, (elevation_at(la, lo), la, lo))
    return best


metres, lat, lon = summit(45.9763, 7.6586)     # the Matterhorn
print(f"summit: {metres} m at {lat:.5f}, {lon:.5f}  (surveyed 4478 m)")

## A profile along a line

Sampling between two points is just a loop — the interesting part is that after the
first call the tiles are local, so a hundred samples cost nothing extra.

In [ ]:
def profile(start, end, samples=40):
    """Elevations along a straight line in lat/lon, which is close enough over a few km."""
    (lat1, lon1), (lat2, lon2) = start, end
    for i in range(samples + 1):
        t = i / samples
        lat = lat1 + (lat2 - lat1) * t
        lon = lon1 + (lon2 - lon1) * t
        yield lat, lon, elevation_at(lat, lon)


# Zermatt village up to the Matterhorn.
points = list(profile((46.0207, 7.7491), (45.9763, 7.6586), samples=40))
low = min(p[2] for p in points)
high = max(p[2] for p in points)
for lat, lon, metres in points:
    bar = "#" * int(40 * (metres - low) / max(1, high - low))
    print(f"{metres:>5} m |{bar}")

If matplotlib happens to be installed, the same data plots in one line. It is not a
dependency of `peaknav`, so this cell is allowed to be skipped.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed - skipping the plot (pip install matplotlib)")
else:
    plt.figure(figsize=(9, 3))
    plt.plot([p[2] for p in points])
    plt.title("Zermatt to the Matterhorn")
    plt.ylabel("metres")
    plt.xlabel("sample")
    plt.grid(alpha=0.3)
    plt.show()

## Underneath

The dataset is a slippy-map tree of tiles, and the elevation is encoded across two
images. `tile_xy` says which tile covers a point; `decode_elevation` turns a pair of
encoded values back into metres. Both are exported so the encoding is inspectable rather
than magic.

In [ ]:
from peaknav.terrain import DATASET_URL, decode_elevation, tile_xy

print("tile covering the Matterhorn:", tile_xy(45.9763, 7.6586))
print("dataset:", DATASET_URL[:70], "...")

# The PNG sample names a 1024 m band (128 is the one starting at sea level) and the JPEG
# the position inside it, in 4 m steps - flipped in odd-numbered bands.
print("decode_elevation(0, 128)   =", decode_elevation(0, 128), "m   (sea level)")
print("decode_elevation(100, 128) =", decode_elevation(100, 128), "m   (100 steps of 4 m)")
print("decode_elevation(161, 129) =", decode_elevation(161, 129), "m   (band 1, flipped)")

### Ocean, and other places with no tile

Where the dataset has no tile at all — open sea — the answer is 0 rather than an error,
which keeps a sweep over a coastline from having to catch exceptions.

In [ ]:
elevation_at(0.0, -30.0)    # middle of the Atlantic